In [4]:
import warnings
warnings.simplefilter("ignore")
import numpy as np
import pandas as pd 
from lightkurve import LightCurve
from scipy.signal import find_peaks
import matplotlib.pyplot as plt
from lightkurve import search_lightcurve 
import lightkurve as lk

for att in ['axes.labelsize', 'axes.titlesize', 'legend.fontsize',
            'legend.fontsize', 'xtick.labelsize', 'ytick.labelsize']:
    plt.rcParams[att] = 15

search_result = search_lightcurve("AU Mic") 
print(search_result)
lc2min = search_result[3].download() 
lc2 = lc2min.remove_nans().remove_outliers()
lc2 = lc2[lc2.quality == 0]
lc2_m = lc2.normalize().remove_nans()

x2min = np.ascontiguousarray(lc2_m.time.value, dtype=np.float64) 
y2min = np.ascontiguousarray(lc2_m.flux, dtype=np.float64)
yerr2min = np.ascontiguousarray(lc2_m.flux_err, dtype=np.float64)
lcquality = np.ascontiguousarray(lc2_m.quality, dtype=np.float64)

SearchResult containing 17 data products.

 #     mission     year       author      exptime target_name distance
                                             s                 arcsec 
--- -------------- ---- ----------------- ------- ----------- --------
  0 TESS Sector 01 2018              SPOC     120   441420236      0.0
  1 TESS Sector 27 2020              SPOC      20   441420236      0.0
  2 TESS Sector 27 2020              SPOC     120   441420236      0.0
  3 TESS Sector 95 2025              SPOC      20   441420236      0.0
  4 TESS Sector 95 2025              SPOC     120   441420236      0.0
  5 TESS Sector 01 2018         TESS-SPOC    1800   441420236      0.0
  6 TESS Sector 27 2020         TESS-SPOC     600   441420236      0.0
  7 TESS Sector 01 2018               QLP    1800   441420236      0.0
  8 TESS Sector 27 2020               QLP     600   441420236      0.0
  9 TESS Sector 95 2025               QLP     200   441420236      0.0
 10 TESS Sector 27 2020           

In [5]:

# ============================================================
# PASSO 2: CARREGAR MÁSCARA DE FLARES E TRÂNSITOS (TXT EDITÁVEL)
# ============================================================
print("\n" + "="*60)
print("ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# Arquivo editável com colunas: t_ini, t_fim
txt_path = "intervalos_flares_transitos_editavel.txt"

try:
    # Lê como CSV simples (separado por vírgula), mas em arquivo .txt
    df_intervalos = pd.read_csv(txt_path, sep=",", comment="#")

    # Validação básica das colunas esperadas
    colunas_esperadas = {"t_ini", "t_fim"}
    if not colunas_esperadas.issubset(df_intervalos.columns):
        raise ValueError(f"Arquivo deve conter as colunas {colunas_esperadas}. Colunas encontradas: {set(df_intervalos.columns)}")

    # Converte para lista de pares [inicio, fim]
    mascara_flares_list = df_intervalos[["t_ini", "t_fim"]].dropna().values.tolist()

    print(f"✓ {len(mascara_flares_list)} intervalo(s) carregado(s) de '{txt_path}':")
    for i, (ini, fim) in enumerate(mascara_flares_list, start=1):
        print(f"  {i}. [{ini:.6f}, {fim:.6f}]")

except Exception as e:
    print(f"✗ Erro ao carregar '{txt_path}': {e}")
    print("→ Usando lista vazia.")
    mascara_flares_list = []

# Criar máscara booleana
mascara_flares = np.zeros(len(t), dtype=bool)
for ini, fim in mascara_flares_list:
    mascara_flares |= (t >= ini) & (t <= fim)

mask_good_flares = ~mascara_flares

print(f"\nMáscara criada:")
print(f"  Pontos excluídos (flares/trânsitos): {mascara_flares.sum()}")
print(f"  Pontos para ajuste: {mask_good_flares.sum()}")



ENTRADA: Carregando intervalos de FLARES e TRÂNSITOS do TXT
✓ 26 intervalo(s) carregado(s) de 'intervalos_flares_transitos_editavel.txt':
  1. [3883.239461, 3883.334868]
  2. [3884.751868, 3884.843937]
  3. [3885.024950, 3885.109542]
  4. [3885.549959, 3885.617899]
  5. [3886.130183, 3886.411659]
  6. [3888.081298, 3888.115210]
  7. [3888.784350, 3888.805991]
  8. [3889.179949, 3889.251286]
  9. [3889.826802, 3889.854535]
  10. [3891.013277, 3891.076262]
  11. [3891.741181, 3891.795356]
  12. [3891.844004, 3891.908792]
  13. [3891.950028, 3892.001562]
  14. [3892.631057, 3892.697367]
  15. [3892.812488, 3892.900441]
  16. [3893.017865, 3893.101673]
  17. [3893.403291, 3893.458281]
  18. [3896.687407, 3896.849926]
  19. [3897.567644, 3897.779646]
  20. [3899.153300, 3899.218797]
  21. [3899.542373, 3899.616046]
  22. [3900.664900, 3900.748400]
  23. [3900.906700, 3900.408000]
  24. [3904.512267, 3904.614076]
  25. [3904.782061, 3904.847219]
  26. [3905.340994, 3905.482584]

Máscara cri

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# PASSO 3: AJUSTE ROTACIONAL BASEADO EM CADÊNCIA + AJUSTES MANUAIS
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# FUNÇÕES PRINCIPAIS
# ============================================================

def gerar_modelo_rotacional_cadencia(t, f, mask_good_flares, cadencia_s=20, janela_horas=10, sigma_clip_val=1.5):
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    pontos_por_hora = 3600 / cadencia_s
    janela_pontos = int(janela_horas * pontos_por_hora)
    
    sigma_pontos = janela_pontos / 8
    
    gap_minimo_minutos = 20
    limite_gap = gap_minimo_minutos / (24 * 60)
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]
    
    modelo_final = np.zeros_like(f)
    
    print(f"\n[Filtro Base Global] Janela: {janela_horas}h | Sigma Clip: {sigma_clip_val}")
    
    for i0, i1 in zip(seg_inicios, seg_fins):
        t_seg = t[i0:i1]
        f_seg = f[i0:i1]
        mask_seg = mask_good_flares[i0:i1]
        
        if len(t_seg) < janela_pontos / 4:
            modelo_final[i0:i1] = np.nanmedian(f_seg)
            continue
            
        f_limpo = np.copy(f_seg)
        
        for iteracao in range(10):
            temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
            residuo = f_seg - temp_smooth
            
            clipped = sigma_clip(
                residuo, 
                sigma_lower=10.0,
                sigma_upper=sigma_clip_val, 
                maxiters=1, 
                cenfunc='median', 
                stdfunc='mad_std'
            )            
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            f_limpo[~mask_seg] = temp_smooth[~mask_seg] 
            
        modelo_final[i0:i1] = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        
    return modelo_final

def ajustar_trecho_especifico(t, f, mask_good_flares, t_inicio, t_fim, 
                              cadencia_s=20, janela_horas=5, 
                              sigma_upper=2.0, sigma_lower=3.0, iteracoes=6):
    """
    Usa a EXATA MESMA LÓGICA do modelo global, mas com parâmetros finos 
    aplicados apenas a um recorte, usando margem de segurança.
    """
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    buffer_dias = (janela_horas * 1.5) / 24.0 
    idx_calc = np.where((t >= t_inicio - buffer_dias) & (t <= t_fim + buffer_dias))[0]
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    
    if len(idx_alvo) == 0:
        return idx_alvo, np.array([])

    t_calc = t[idx_calc]
    f_calc = f[idx_calc]
    mask_calc = mask_good_flares[idx_calc]
    
    pontos_por_hora = 3600 / cadencia_s
    sigma_pontos = (janela_horas * pontos_por_hora) / 8
    
    f_limpo = np.copy(f_calc)
    
    for _ in range(iteracoes):
        temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        residuo = f_calc - temp_smooth
        
        clipped = sigma_clip(residuo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, 
                             maxiters=1, cenfunc='median', stdfunc='mad_std')
        
        mascara_combinada = (~clipped.mask) & mask_calc
        t_bons = t_calc[mascara_combinada]
        f_bons = f_limpo[mascara_combinada]
        
        if len(t_bons) > 2:
            f_limpo = np.interp(t_calc, t_bons, f_bons)
        else:
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            
    modelo_calc = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
    
    inicio_corte = np.where(idx_calc == idx_alvo[0])[0][0]
    fim_corte = np.where(idx_calc == idx_alvo[-1])[0][0] + 1
    
    return idx_alvo, modelo_calc[inicio_corte:fim_corte]

def costurar_bordas(t, modelo, bordas, tamanho_janela=150, sigma_gauss=0):
    from scipy.ndimage import gaussian_filter1d
    modelo_costurado = np.copy(modelo).astype(float)
    n = len(t)
    
    for borda_idx in sorted(bordas):
        inicio = max(0, borda_idx - tamanho_janela)
        fim = min(n - 1, borda_idx + tamanho_janela)
        if (fim - inicio) < 40: continue
            
        meia_zona = (fim - inicio) // 4
        fim_esq = max(inicio + 5, borda_idx - meia_zona)
        ini_dir = min(fim - 5, borda_idx + meia_zona)
        
        idx_esq = np.arange(inicio, fim_esq)
        idx_dir = np.arange(ini_dir, fim + 1)
        if len(idx_esq) < 10 or len(idx_dir) < 10: continue
            
        t_centro = t[borda_idx]
        poly_esq = np.polyfit(t[idx_esq] - t_centro, modelo_costurado[idx_esq], deg=2)
        poly_dir = np.polyfit(t[idx_dir] - t_centro, modelo_costurado[idx_dir], deg=2)
        
        zona_miolo = np.arange(fim_esq, ini_dir + 1)
        t_miolo = t[zona_miolo] - t_centro
        
        pred_esq = np.polyval(poly_esq, t_miolo)
        pred_dir = np.polyval(poly_dir, t_miolo)
        
        x_norm = (t[zona_miolo] - t[fim_esq]) / (t[ini_dir] - t[fim_esq] + 1e-10)
        x_norm = np.clip(x_norm, 0, 1)
        peso = 3 * x_norm**2 - 2 * x_norm**3 
        
        modelo_costurado[zona_miolo] = (1 - peso) * pred_esq + peso * pred_dir
        
    if sigma_gauss > 0:
        dt_mediano = np.median(np.diff(t))
        quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [n]):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)
                
    return modelo_costurado

def costurar_com_gaps(t, modelo, bordas, limite_gap_fator=5, tamanho_janela=32, sigma_gauss=15):
    from scipy.ndimage import gaussian_filter1d
    dt_mediano = np.median(np.diff(t))
    limite_gap = dt_mediano * limite_gap_fator
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]

    # Chama a costura original, mas sem aplicar o gauss globalmente
    modelo_costurado = costurar_bordas(
        t, modelo, bordas,
        tamanho_janela=tamanho_janela,
        sigma_gauss=0  # sem gauss aqui para não vazar nos gaps
    )

    # Aplica a suavização gaussiana APENAS dentro dos blocos contínuos de dados
    if sigma_gauss > 0:
        for i0, i1 in zip(seg_inicios, seg_fins):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)

    return modelo_costurado

# ============================================================
# GERANDO A BASE OFICIAL GLOBAL
# ============================================================

# Modelo global com parâmetros que funcionam bem para a maior parte
modelo_manchas = gerar_modelo_rotacional_cadencia(
    t, f, mask_good_flares, 
    cadencia_s=20, 
    janela_horas=10, 
    sigma_clip_val=1.8
)

bordas_indices = set()

# ============================================================
# REGIÕES DE AJUSTE FINO (Usando a MESMA LÓGICA da base)
# ============================================================
# Formato: [t_inicio, t_fim, janela_horas, sigma_upper, sigma_lower, iteracoes]

regioes_ajuste_local = [
    # Ajuste fino para a região solicitada:
    # Deixei a janela com 3 horas (mais flexível) e o corte superior rigoroso (1.2)
    #[3899.9880, 3900.9960, 10.0, 1.2, 3.0, 8]
    [3884.6271, 3884.9503, 10.0, 1.2, 3.0, 12],

    [3885.9927, 3886.4632, 10.0, 5, 5.0, 12],
    
    [3902.9317, 3903.3155, 12, 5, 5.0, 12],

    
    # 2. Região das Imagens 3 e 4 (Dias 3899.8 a 3901.0 - Flares GIGANTES)
    # Aqui a gravidade tem que ser extrema. sigma_upper em 0.5 força a linha pro chão.
    #[3899.8, 3901.4063, 10, 1.2, 3.0, 12]
]

for ini, fim, janela_h, sig_up, sig_low, iters in regioes_ajuste_local:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx): continue
    
    # Processando o ajuste local usando a sua função física com parâmetros finos
    idx_alvo, mod_local = ajustar_trecho_especifico(
        t, f, mask_good_flares, 
        t_inicio=ini, 
        t_fim=fim, 
        janela_horas=janela_h,      # Ex: 5.0 para seguir melhor as variações locais
        sigma_upper=sig_up,         # Ex: 1.2 para mutilar a base do flare local
        sigma_lower=sig_low,        # Ex: 3.0 para não perder a referência de baixo
        iteracoes=iters             # Ex: 8 para nivelar bem a curva
    )
    
    if len(idx_alvo) > 0:
        modelo_manchas[idx_alvo] = mod_local
        bordas_indices.add(idx_alvo[0])
        bordas_indices.add(idx_alvo[-1])

# ============================================================
# COSTURA FINAL DAS REGIÕES LOCAIS
# ============================================================

if len(bordas_indices) > 0:
    print(f"\nCosturando {len(bordas_indices)} bordas de ajustes finos locais...")
    modelo_manchas_suave = costurar_bordas(
        t, modelo_manchas, sorted(list(bordas_indices)),
        tamanho_janela=150, 
        sigma_gauss=20  
    )
else:
    print("\nNenhum ajuste local aplicado. Usando o modelo global liso.")
    from scipy.ndimage import gaussian_filter1d
    modelo_manchas_suave = modelo_manchas
    
    dt_mediano = np.median(np.diff(t))
    quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
    for i0, i1 in zip([0] + quebras, quebras + [len(t)]):
        if i1 - i0 > 1:
            modelo_manchas_suave[i0:i1] = gaussian_filter1d(modelo_manchas_suave[i0:i1], sigma=20)


# ============================================================
# RESÍDUO E GRÁFICOS
# ============================================================

%matplotlib qt

# ============================================================
# ESTE COMANDO FAZ O MATPLOTLIB ABRIR EM UMA JANELA SEPARADA
# (Deve ser a primeira linha da célula)
# ============================================================
%matplotlib qt

import matplotlib.pyplot as plt

# ============================================================
# RESÍDUO E GRÁFICOS (MATPLOTLIB EM JANELA EXTERNA)
# ============================================================

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados')
ax1.plot(t, modelo_manchas_suave, color='red', lw=2.2, label='Modelo Final')

# Pinta de azul as regiões onde o ajuste local fino foi aplicado
for ini, fim, *_ in regioes_ajuste_local:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.10, label='_ajuste_local')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_ajuste_local:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.10)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais

[Filtro Base Global] Janela: 10h | Sigma Clip: 1.8

Costurando 6 bordas de ajustes finos locais...

OK — Ajuste concluído.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# PASSO 3: AJUSTE ROTACIONAL BASEADO EM CADÊNCIA + AJUSTES MANUAIS
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# FUNÇÕES PRINCIPAIS
# ============================================================

def gerar_modelo_rotacional_cadencia(t, f, mask_good_flares, cadencia_s=20, janela_horas=10, sigma_clip_val=1.5):
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    pontos_por_hora = 3600 / cadencia_s
    janela_pontos = int(janela_horas * pontos_por_hora)
    
    sigma_pontos = janela_pontos / 8
    
    gap_minimo_minutos = 20
    limite_gap = gap_minimo_minutos / (24 * 60)
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]
    
    modelo_final = np.zeros_like(f)
    
    print(f"\n[Filtro Base Global] Janela: {janela_horas}h | Sigma Clip: {sigma_clip_val}")
    
    for i0, i1 in zip(seg_inicios, seg_fins):
        t_seg = t[i0:i1]
        f_seg = f[i0:i1]
        mask_seg = mask_good_flares[i0:i1]
        
        if len(t_seg) < janela_pontos / 4:
            modelo_final[i0:i1] = np.nanmedian(f_seg)
            continue
            
        f_limpo = np.copy(f_seg)
        
        for iteracao in range(6):
            temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
            residuo = f_seg - temp_smooth
            
            clipped = sigma_clip(
                residuo, 
                sigma_lower=10.0,
                sigma_upper=sigma_clip_val, 
                maxiters=1, 
                cenfunc='median', 
                stdfunc='mad_std'
            )            
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            f_limpo[~mask_seg] = temp_smooth[~mask_seg] 
            
        modelo_final[i0:i1] = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        
    return modelo_final

def ajustar_trecho_especifico(t, f, mask_good_flares, t_inicio, t_fim, 
                              cadencia_s=20, janela_horas=5, 
                              sigma_upper=2.0, sigma_lower=3.0, iteracoes=6):
    """
    Usa a EXATA MESMA LÓGICA do modelo global, mas com parâmetros finos 
    aplicados apenas a um recorte, usando margem de segurança.
    """
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    buffer_dias = (janela_horas * 1.5) / 24.0 
    idx_calc = np.where((t >= t_inicio - buffer_dias) & (t <= t_fim + buffer_dias))[0]
    idx_alvo = np.where((t >= t_inicio) & (t <= t_fim))[0]
    
    if len(idx_alvo) == 0:
        return idx_alvo, np.array([])

    t_calc = t[idx_calc]
    f_calc = f[idx_calc]
    mask_calc = mask_good_flares[idx_calc]
    
    pontos_por_hora = 3600 / cadencia_s
    sigma_pontos = (janela_horas * pontos_por_hora) / 8
    
    f_limpo = np.copy(f_calc)
    
    for _ in range(iteracoes):
        temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
        residuo = f_calc - temp_smooth
        
        clipped = sigma_clip(residuo, sigma_lower=sigma_lower, sigma_upper=sigma_upper, 
                             maxiters=1, cenfunc='median', stdfunc='mad_std')
        
        mascara_combinada = (~clipped.mask) & mask_calc
        t_bons = t_calc[mascara_combinada]
        f_bons = f_limpo[mascara_combinada]
        
        if len(t_bons) > 2:
            f_limpo = np.interp(t_calc, t_bons, f_bons)
        else:
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            
    modelo_calc = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='reflect')
    
    inicio_corte = np.where(idx_calc == idx_alvo[0])[0][0]
    fim_corte = np.where(idx_calc == idx_alvo[-1])[0][0] + 1
    
    return idx_alvo, modelo_calc[inicio_corte:fim_corte]

def costurar_bordas(t, modelo, bordas, tamanho_janela=150, sigma_gauss=0):
    from scipy.ndimage import gaussian_filter1d
    modelo_costurado = np.copy(modelo).astype(float)
    n = len(t)
    
    for borda_idx in sorted(bordas):
        inicio = max(0, borda_idx - tamanho_janela)
        fim = min(n - 1, borda_idx + tamanho_janela)
        if (fim - inicio) < 40: continue
            
        meia_zona = (fim - inicio) // 4
        fim_esq = max(inicio + 5, borda_idx - meia_zona)
        ini_dir = min(fim - 5, borda_idx + meia_zona)
        
        idx_esq = np.arange(inicio, fim_esq)
        idx_dir = np.arange(ini_dir, fim + 1)
        if len(idx_esq) < 10 or len(idx_dir) < 10: continue
            
        t_centro = t[borda_idx]
        poly_esq = np.polyfit(t[idx_esq] - t_centro, modelo_costurado[idx_esq], deg=2)
        poly_dir = np.polyfit(t[idx_dir] - t_centro, modelo_costurado[idx_dir], deg=2)
        
        zona_miolo = np.arange(fim_esq, ini_dir + 1)
        t_miolo = t[zona_miolo] - t_centro
        
        pred_esq = np.polyval(poly_esq, t_miolo)
        pred_dir = np.polyval(poly_dir, t_miolo)
        
        x_norm = (t[zona_miolo] - t[fim_esq]) / (t[ini_dir] - t[fim_esq] + 1e-10)
        x_norm = np.clip(x_norm, 0, 1)
        peso = 3 * x_norm**2 - 2 * x_norm**3 
        
        modelo_costurado[zona_miolo] = (1 - peso) * pred_esq + peso * pred_dir
        
    if sigma_gauss > 0:
        dt_mediano = np.median(np.diff(t))
        quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [n]):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)
                
    return modelo_costurado


# ============================================================
# GERANDO A BASE OFICIAL GLOBAL
# ============================================================

# Modelo global com parâmetros que funcionam bem para a maior parte
modelo_manchas = gerar_modelo_rotacional_cadencia(
    t, f, mask_good_flares, 
    cadencia_s=20, 
    janela_horas=10, 
    sigma_clip_val=2.2
)

bordas_indices = set()

# ============================================================
# REGIÕES DE AJUSTE FINO (Usando a MESMA LÓGICA da base)
# ============================================================
# Formato: [t_inicio, t_fim, janela_horas, sigma_upper, sigma_lower, iteracoes]

regioes_ajuste_local = [
    # Ajuste fino para a região solicitada:
    # Deixei a janela com 5 horas (mais flexível) e o corte superior rigoroso (1.2)
    [3899.9880, 3900.9960, 5.0, 1.2, 3.0, 8]
]

for ini, fim, janela_h, sig_up, sig_low, iters in regioes_ajuste_local:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx): continue
    
    # Processando o ajuste local usando a sua função física com parâmetros finos
    idx_alvo, mod_local = ajustar_trecho_especifico(
        t, f, mask_good_flares, 
        t_inicio=ini, 
        t_fim=fim, 
        janela_horas=janela_h,      # Ex: 5.0 para seguir melhor as variações locais
        sigma_upper=sig_up,         # Ex: 1.2 para mutilar a base do flare local
        sigma_lower=sig_low,        # Ex: 3.0 para não perder a referência de baixo
        iteracoes=iters             # Ex: 8 para nivelar bem a curva
    )
    
    if len(idx_alvo) > 0:
        modelo_manchas[idx_alvo] = mod_local
        bordas_indices.add(idx_alvo[0])
        bordas_indices.add(idx_alvo[-1])

# ============================================================
# COSTURA FINAL DAS REGIÕES LOCAIS
# ============================================================

if len(bordas_indices) > 0:
    print(f"\nCosturando {len(bordas_indices)} bordas de ajustes finos locais...")
    modelo_manchas_suave = costurar_bordas(
        t, modelo_manchas, sorted(list(bordas_indices)),
        tamanho_janela=150, 
        sigma_gauss=20  
    )
else:
    print("\nNenhum ajuste local aplicado. Usando o modelo global liso.")
    from scipy.ndimage import gaussian_filter1d
    modelo_manchas_suave = modelo_manchas
    
    dt_mediano = np.median(np.diff(t))
    quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
    for i0, i1 in zip([0] + quebras, quebras + [len(t)]):
        if i1 - i0 > 1:
            modelo_manchas_suave[i0:i1] = gaussian_filter1d(modelo_manchas_suave[i0:i1], sigma=20)


# ============================================================
# RESÍDUO E GRÁFICOS
# ============================================================

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados')
ax1.plot(t, modelo_manchas_suave, color='red', lw=2.2, label='Modelo Final')

# Pinta de azul as regiões onde o ajuste local fino foi aplicado
for ini, fim, *_ in regioes_ajuste_local:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.10, label='_ajuste_local')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_ajuste_local:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.10)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais

[Filtro Base] Janela: 10h | Sigma Clip: 2.2 | Tolerância Gap: 20 min

Costurando 2 bordas de ajustes manuais...

OK — Ajuste concluído.


In [ ]:
# ============================================================
# PASSO 3: AJUSTE ROTACIONAL BASEADO EM CADÊNCIA + AJUSTES MANUAIS
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# FUNÇÕES PRINCIPAIS
# ============================================================

def gerar_modelo_rotacional_cadencia(t, f, mask_good_flares, cadencia_s=20, janela_horas=10, sigma_clip_val=2.5):
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    pontos_por_hora = 3600 / cadencia_s
    janela_pontos = int(janela_horas * pontos_por_hora)
    sigma_pontos = janela_pontos / 6
    
    dt_mediano = np.median(np.diff(t))
    limite_gap = dt_mediano * 5
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]
    
    modelo_final = np.zeros_like(f)
    
    print(f"\n[Cadência 20s] Janela de Suavização: {janela_horas} horas ({janela_pontos} pontos).")
    
    for i0, i1 in zip(seg_inicios, seg_fins):
        t_seg = t[i0:i1]
        f_seg = f[i0:i1]
        mask_seg = mask_good_flares[i0:i1]
        
        if len(t_seg) < 50:
            modelo_final[i0:i1] = np.nanmedian(f_seg)
            continue
            
        f_limpo = np.copy(f_seg)
        for iteracao in range(4):
            temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
            residuo = f_seg - temp_smooth
            clipped = sigma_clip(residuo, sigma=sigma_clip_val, maxiters=1, cenfunc='median', stdfunc='mad_std')
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            f_limpo[~mask_seg] = temp_smooth[~mask_seg] 
            
        modelo_final[i0:i1] = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
        
    return modelo_final

def costurar_bordas(t, modelo, bordas, tamanho_janela=150, sigma_gauss=0):
    from scipy.ndimage import gaussian_filter1d
    modelo_costurado = np.copy(modelo).astype(float)
    n = len(t)
    
    for borda_idx in sorted(bordas):
        inicio = max(0, borda_idx - tamanho_janela)
        fim = min(n - 1, borda_idx + tamanho_janela)
        if (fim - inicio) < 40: continue
            
        meia_zona = (fim - inicio) // 4
        fim_esq = max(inicio + 5, borda_idx - meia_zona)
        ini_dir = min(fim - 5, borda_idx + meia_zona)
        
        idx_esq = np.arange(inicio, fim_esq)
        idx_dir = np.arange(ini_dir, fim + 1)
        if len(idx_esq) < 10 or len(idx_dir) < 10: continue
            
        t_centro = t[borda_idx]
        poly_esq = np.polyfit(t[idx_esq] - t_centro, modelo_costurado[idx_esq], deg=2)
        poly_dir = np.polyfit(t[idx_dir] - t_centro, modelo_costurado[idx_dir], deg=2)
        
        zona_miolo = np.arange(fim_esq, ini_dir + 1)
        t_miolo = t[zona_miolo] - t_centro
        
        pred_esq = np.polyval(poly_esq, t_miolo)
        pred_dir = np.polyval(poly_dir, t_miolo)
        
        x_norm = (t[zona_miolo] - t[fim_esq]) / (t[ini_dir] - t[fim_esq] + 1e-10)
        x_norm = np.clip(x_norm, 0, 1)
        peso = 3 * x_norm**2 - 2 * x_norm**3 
        
        modelo_costurado[zona_miolo] = (1 - peso) * pred_esq + peso * pred_dir
        
    if sigma_gauss > 0:
        dt_mediano = np.median(np.diff(t))
        quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
        for i0, i1 in zip([0] + quebras, quebras + [n]):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)
                
    return modelo_costurado


# ============================================================
# GERANDO A BASE OFICIAL LISINHA (O FIM DO N_SEGMENTOS_AUTO!)
# ============================================================

# Aqui nós usamos a cadência para criar uma base contínua e perfeita
modelo_manchas = gerar_modelo_rotacional_cadencia(
    t, f, mask_good_flares, 
    cadencia_s=20, 
    janela_horas=10, 
    sigma_clip_val=2.2
)

bordas_indices = set()

# ============================================================
# REGIÕES MANUAIS (Só ative se a base global falhar em algum ponto)
# ============================================================
# Se a curva global ficar perfeita, você pode até deixar essas listas vazias []

ajustes_manuais = [] # Deixei vazio por padrão para testar a base pura

regioes_spline = [
    [3899.9880, 3900.996, 100, 4, 0.9, 1.2]
]

# (Aqui iriam as lógicas do fitting_spline se você for usar a região acima. 
# Importe a função fitting_spline como estava no seu código original caso 
# queira processar splines por cima da curva base).

# Exemplo de como aplicar a região spline por cima da base lisa:
for ini, fim, nbins, k, fator_s, sigma_clip_int in regioes_spline:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx): continue
    
    # modelo_manchas[idx] = fitting_spline(...) # Descomente e coloque sua função fitting_spline aqui
    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# ============================================================
# COSTURA FINAL 
# (Só fará algo se houver bordas geradas pelas regiões manuais)
# ============================================================

if len(bordas_indices) > 0:
    print(f"\nCosturando {len(bordas_indices)} bordas de ajustes manuais...")
    modelo_manchas_suave = costurar_bordas(
        t, modelo_manchas, sorted(list(bordas_indices)),
        tamanho_janela=150, 
        sigma_gauss=20  
    )
else:
    print("\nNenhum ajuste manual aplicado. Usando a base global puramente liso.")
    # Se não há bordas, o modelo base já é o modelo final
    from scipy.ndimage import gaussian_filter1d
    modelo_manchas_suave = modelo_manchas
    
    # Aplica só um polimento final no modelo global
    dt_mediano = np.median(np.diff(t))
    quebras = list(np.where(np.diff(t) > dt_mediano * 5)[0] + 1)
    for i0, i1 in zip([0] + quebras, quebras + [len(t)]):
        if i1 - i0 > 1:
            modelo_manchas_suave[i0:i1] = gaussian_filter1d(modelo_manchas_suave[i0:i1], sigma=20)


# ============================================================
# RESÍDUO E GRÁFICOS
# ============================================================

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados')
ax1.plot(t, modelo_manchas_suave, color='red', lw=2.2, label='Modelo Final')

for ini, fim, *_ in regioes_spline:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.10, label='_spline')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_spline:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.10)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Modelo Rotacional Físico (Global) + Correções Locais

[Filtro Base] Janela: 10h | Sigma Clip: 2.2 | Tolerância Gap: 20 min

Nenhum ajuste manual aplicado. Usando a base global puramente liso.

OK — Ajuste concluído.


In [28]:
from astropy.stats import sigma_clip
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
from lightkurve import LightCurve

from scipy.interpolate import CubicSpline
from scipy.ndimage import gaussian_filter1d

# ============================================================
# PASSO 3: AJUSTE POLINOMIAL + FLATTEN LOCAL + SPLINE LOCAL
# ============================================================

print("\n" + "="*60)
print("AJUSTE: Polinomial com Segmentos + Spline em Regiões")
print("="*60)

t = np.array(lc2_m.time.value)
f = np.array(lc2_m.flux)

# ============================================================
# REGIÕES MANUAIS (poly / flatten)
# ============================================================

ajustes_manuais = [
    [3883.0050, 3883.4554, "poly", 4, 2.5],
    [3883.1409, 3883.7849, "poly", 4, 2.5],
    [3883.8264, 3884.0955, "poly", 4, 2.5],
    [3884.2828, 3885.3000, "poly", 4, 2.5],
    [3885.2839, 3885.4326, "poly", 4, 2.5],
    [3885.6205, 3885.7794, "poly", 2, 2.5],
    [3892.6280, 3892.7040, "poly", 1, 2.5],
    [3894.2089, 3896.5099, "poly", 4, 2.0],

    #[3899.86607, 3900.8691, "poly", 6, 2.5],
    #[3900.8273, 3900.8691, "flatten", None, None],
]

# ============================================================
# REGIÕES SPLINE  <-- ADICIONE / EDITE AQUI
# Formato: [t_inicio, t_fim, nbins, grau_k, fator_s, sigma_clip_interno]
# ============================================================

regioes_spline = [
    # [t_ini,      t_fim,   nbins, k, fator_s, sigma_clip_int]
    #[3899.86607, 3900.8691,  190,  4, 0.0001,  1.5],  # região com flares
    #[3900.8273,  3901.44,     60,  4, 0.001,   1.5],  # região mais tranquila
    # [3905.0,   3907.00,     60,  3, 0.01,    2.2],  # descomente para mais
    [3899.9880, 3900.996, 100, 4, 0.9, 1.2]    #[3894.1100, 3896.5099, 40, 2, 0.05, 1.0],  # AJUSTADO: menos bins, grau menor, mais suavização

    #[3899.501, 3900.826,   60,  7,  0.01,   1.5],  # mais suave, ignora flares

    #[3900.8273,  3901.44,     30,  3,  0.01,   1.5],
]


# ============================================================
# REGIÕES FLATTEN LOCAL  <-- ADICIONE / EDITE AQUI
# Formato: [t_inicio, t_fim, window_length, polyorder, sigma, break_tolerance, niters]
# Exemplo: [3900.8273, 3900.9512, 320, 3, 2.5, 10, 4]
# ============================================================

regioes_flatten = [
    #[3900.5909, 3900.9512, 280, 1, 1.5, 1, 4],
    # [3887.1500, 3888.4400, 280, 2, 2.2, 8, 5],
]

# ============================================================
# FLATTEN GLOBAL (fallback para uso em ajustes_manuais do tipo flatten)
# ============================================================

flcd, trend = lc2_m.flatten(
    window_length=320,
    polyorder=3,
    return_trend=True,
    break_tolerance=10,
    niters=4,
    sigma=2.5,
    mask=mask_good_flares
)

# ============================================================
# AJUSTE AUTOMÁTICO
# ============================================================

N_SEGMENTOS_AUTO = 10
GRAU_AUTO        = 4
SIGMA_AUTO       = 2.9

# ============================================================
# FUNÇÕES
# ============================================================

def fitting_segment(t_seg, f_seg, mask_seg, deg, sigma):
    if len(t_seg) < deg + 2:
        return np.full_like(f_seg, np.nanmedian(f_seg))
    t_mid = np.median(t_seg)
    t_s   = t_seg - t_mid
    good  = mask_seg.copy()
    modelo = np.full_like(f_seg, np.nan)
    for _ in range(5):
        if np.sum(good) < deg + 2:
            break
        coef = np.polyfit(t_s[good], f_seg[good], deg=deg)
        modelo_good = np.polyval(coef, t_s[good])
        modelo = np.interp(t_s, t_s[good], modelo_good)
        resid = f_seg - modelo
        clipped = sigma_clip(resid[good], sigma=sigma, maxiters=1)
        if clipped.mask is np.ma.nomask:
            break
        good[np.where(good)[0]] = ~clipped.mask
    if np.all(np.isnan(modelo)):
        modelo = np.full_like(f_seg, np.nanmedian(f_seg))
    return modelo


def fitting_spline(t_seg, f_seg, mask_seg, nbins, k, fator_s, sigma_clip_interno=1.8, clip_iters=5):
    t_ok = t_seg[mask_seg]
    f_ok = f_seg[mask_seg]

    if len(t_ok) < k + 2:
        print("    [spline] pontos insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    # clip robusto a outliers (mad_std nao eh inflado pela cauda do flare)
    if sigma_clip_interno is not None and len(f_ok) > k + 2:
        cl = sigma_clip(
            f_ok, sigma=sigma_clip_interno, maxiters=clip_iters,
            cenfunc='median', stdfunc='mad_std'
        )
        t_ok = t_ok[~cl.mask]
        f_ok = f_ok[~cl.mask]

    if len(t_ok) < k + 2:
        print("    [spline] pontos insuficientes apos sigma_clip, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    edges = np.linspace(t_ok.min(), t_ok.max(), nbins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    bin_t, bin_f = [], []
    for i in range(nbins):
        in_bin = (t_ok >= edges[i]) & (t_ok < edges[i + 1])
        if np.sum(in_bin) >= 1:
            bin_t.append(centers[i])
            bin_f.append(np.nanmedian(f_ok[in_bin]))

    bin_t = np.array(bin_t)
    bin_f = np.array(bin_f)

    if len(bin_t) < k + 2:
        print("    [spline] bins insuficientes, usando mediana.")
        return np.full_like(f_seg, np.nanmedian(f_seg))

    # segunda passada: remove BINS anomalos (ex: cauda de flare que escapou do filtro ponto a ponto)
    if len(bin_f) > k + 3:
        cl_bins = sigma_clip(bin_f, sigma=2.0, maxiters=3, cenfunc='median', stdfunc='mad_std')
        if np.sum(~cl_bins.mask) >= k + 2:
            n_removidos = np.sum(cl_bins.mask)
            if n_removidos > 0:
                print(f"    [spline] {n_removidos} bin(s) anomalo(s) removido(s) antes do ajuste.")
            bin_t = bin_t[~cl_bins.mask]
            bin_f = bin_f[~cl_bins.mask]

    s_val = fator_s * len(bin_t) * np.nanvar(bin_f) if fator_s > 0 else 0.0
    spline = UnivariateSpline(bin_t, bin_f, k=k, s=s_val, ext=3)
    modelo = spline(t_seg)

    med = np.nanmedian(f_ok)
    std_ok = np.nanstd(f_ok)
    limite = 6 * std_ok if std_ok > 0 else 6 * abs(med) + 1e-6
    fora = np.abs(modelo - med) > limite
    if np.any(fora):
        print(f"    [spline] {np.sum(fora)} ponto(s) fora do limite seguro, substituindo.")
        bons = ~fora
        if np.sum(bons) >= 2:
            modelo[fora] = np.interp(t_seg[fora], t_seg[bons], modelo[bons])
        else:
            modelo[fora] = med

    return modelo


def costurar_bordas(t, modelo, bordas, dados_observados=None, tamanho_janela=50, sigma_gauss=15):
    """
    Costura usando Extrapolação Local e Mistura Suave (Smoothstep Blending).
    Elimina completamente oscilações (overshoots), degraus e "ondas" causadas por splines.
    """
    from scipy.ndimage import gaussian_filter1d
    
    modelo_costurado = np.copy(modelo).astype(float)
    n = len(t)
    print(f"\nAplicando costura ultra-suave (Blending) em {len(bordas)} emendas...")
    
    # Processa as bordas ordenadas para evitar conflitos de índices
    for borda_idx in sorted(bordas):
        # Define a janela ampla ao redor da emenda
        inicio = max(0, borda_idx - tamanho_janela)
        fim = min(n - 1, borda_idx + tamanho_janela)
        
        if (fim - inicio) < 40:
            continue
            
        # Zona de transição (o "miolo" onde o corte aconteceu)
        # Usamos 1/4 da janela para cada lado do centro para fazer a fusão suave
        meia_zona = (fim - inicio) // 4
        fim_esq = max(inicio + 5, borda_idx - meia_zona)
        ini_dir = min(fim - 5, borda_idx + meia_zona)
        
        # Regiões estáveis ("âncoras") de cada lado, fora da zona de quebra
        idx_esq = np.arange(inicio, fim_esq)
        idx_dir = np.arange(ini_dir, fim + 1)
        
        if len(idx_esq) < 10 or len(idx_dir) < 10:
            continue
            
        # Ajustamos uma parábola (grau 2) em cada lado para capturar a inclinação real.
        # Polinômios de grau 2 são rígidos e IMUNES a oscilações selvagens.
        grau_ajuste = 3
        
        # Subtraímos o tempo central para estabilidade numérica do ajuste
        t_centro = t[borda_idx]
        poly_esq = np.polyfit(t[idx_esq] - t_centro, modelo_costurado[idx_esq], deg=grau_ajuste)
        poly_dir = np.polyfit(t[idx_dir] - t_centro, modelo_costurado[idx_dir], deg=grau_ajuste)
        
        # O miolo que será recalculado e fundido de forma contínua
        zona_miolo = np.arange(fim_esq, ini_dir + 1)
        t_miolo = t[zona_miolo] - t_centro
        
        # Extrapola os caminhos naturais que a curva de luz faria se não houvesse a quebra
        pred_esq = np.polyval(poly_esq, t_miolo)
        pred_dir = np.polyval(poly_dir, t_miolo)
        
        # Cria uma curva de peso Smoothstep (C1 contínuo, sem quinas na derivada)
        # Vai de 0.0 (totalmente a esquerda) a 1.0 (totalmente a direita)
        x_norm = (t[zona_miolo] - t[fim_esq]) / (t[ini_dir] - t[fim_esq] + 1e-10)
        x_norm = np.clip(x_norm, 0, 1)
        
        # Função Smoothstep: 3x² - 2x³ (garante transição suave nas pontas)
        peso = 3 * x_norm**2 - 2 * x_norm**3 
        
        # Combinação linear ponderada das duas tendências estáveis
        modelo_costurado[zona_miolo] = (1 - peso) * pred_esq + peso * pred_dir
        
    # Filtro Gaussiano Final por segmento para limar qualquer micro-ruído de alta frequência
    if sigma_gauss > 0:
        dt_mediano = np.median(np.diff(t))
        limite_gap = dt_mediano * 5
        quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
        seg_inicios = [0] + quebras
        seg_fins    = quebras + [n]
        
        for i0, i1 in zip(seg_inicios, seg_fins):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)
                
    return modelo_costurado


def gerar_modelo_rotacional_cadencia(t, f, mask_good_flares, cadencia_s=20, janela_horas=10, sigma_clip_val=2.5):
    """
    Gera um modelo rotacional puro, ultra-suave, baseado na cadência de 20s.
    Substitui a divisão em 40 pedaços por um filtro robusto global por segmento real.
    """
    from scipy.ndimage import gaussian_filter1d
    from astropy.stats import sigma_clip
    
    # 1. Calcula o tamanho da janela em pontos baseado na física da cadência
    pontos_por_hora = 3600 / cadencia_s
    janela_pontos = int(janela_horas * pontos_por_hora)
    
    # Para o filtro gaussiano, o sigma ideal é cerca de 1/6 da janela total de tempo
    sigma_pontos = janela_pontos / 6
    
    # 2. Identifica apenas GAPS REAIS de observação (onde o telescópio parou)
    dt_mediano = np.median(np.diff(t))
    limite_gap = dt_mediano * 5
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]
    
    modelo_final = np.zeros_like(f)
    
    print(f"\n[Cadência 20s] Janela de Suavização: {janela_horas} horas ({janela_pontos} pontos).")
    print(f"Processando {len(seg_inicios)} segmento(s) contínuo(s) real(is)...")
    
    for i0, i1 in zip(seg_inicios, seg_fins):
        t_seg = t[i0:i1]
        f_seg = f[i0:i1]
        mask_seg = mask_good_flares[i0:i1]
        
        if len(t_seg) < 50:
            modelo_final[i0:i1] = np.nanmedian(f_seg)
            continue
            
        # 3. Limpeza Robusta Iterativa (Garante que os flares não puxem a curva para cima)
        f_limpo = np.copy(f_seg)
        
        # Fazemos 4 passadas para remover os flares e preencher com a tendência suave
        for iteracao in range(4):
            temp_smooth = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
            residuo = f_seg - temp_smooth
            
            # Clip nos pontos que sobressaem da tendência (flares)
            clipped = sigma_clip(residuo, sigma=sigma_clip_val, maxiters=1, cenfunc='median', stdfunc='mad_std')
            
            # Substitui temporariamente os outliers pelo valor do modelo suave
            f_limpo[clipped.mask] = temp_smooth[clipped.mask]
            f_limpo[~mask_seg] = temp_smooth[~mask_seg] # Força o uso da máscara externa também
            
        # 4. Ajuste final sobre os dados perfeitamente limpos
        modelo_final[i0:i1] = gaussian_filter1d(f_limpo, sigma=sigma_pontos, mode='nearest')
        
    return modelo_final

# ============================================================
# SUBSTITUA AS CHAMADAS ANTIGAS POR ESTA LINHA:
# ============================================================

# Gera o modelo perfeitamente liso usando a cadência de 20 segundos
# Gera o modelo perfeitamente liso usando a cadência de 20 segundos
modelo_manchas_suave = gerar_modelo_rotacional_cadencia(
    t, f, mask_good_flares, 
    cadencia_s=20, 
    janela_horas=10,       # 10 horas de largura de filtro garante aspecto rotacional puro
    sigma_clip_val=2.2     # Rejeição agressiva de flares para não deformar a base
)

# Se você ainda tiver regiões manuais muito específicas (como trânsitos ou gaps instrumentais),
# você pode aplicar a função costurar_bordas original APENAS nelas.
# ============================================================
# AJUSTE AUTOMÁTICO BASE
# ============================================================

modelo_manchas = np.zeros_like(f)
bordas_indices = set()
edges_auto = np.linspace(t.min(), t.max(), N_SEGMENTOS_AUTO + 1)
segmentos = [(edges_auto[i], edges_auto[i + 1]) for i in range(N_SEGMENTOS_AUTO)]

print("Ajuste automático...")
for i, (ini, fim) in enumerate(segmentos):
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue
    modelo_manchas[idx] = fitting_segment(
        t[idx], f[idx], mask_good_flares[idx], GRAU_AUTO, SIGMA_AUTO
    )
    if i > 0:
        bordas_indices.add(np.where(idx)[0][0])

# ============================================================
# AJUSTES MANUAIS (poly / flatten)
# ============================================================

print(f"Aplicando {len(ajustes_manuais)} ajustes manuais...")
for ini, fim, metodo, grau, sig in ajustes_manuais:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        continue

    if metodo == "poly":
        modelo_manchas[idx] = fitting_segment(
            t[idx], f[idx], mask_good_flares[idx], grau, sig
        )

    elif metodo == "flatten":
        print(f"  FLATTEN (global) {ini:.4f} - {fim:.4f}")
        modelo_manchas[idx] = trend.flux.value[idx]

        idx_inicio = np.where(idx)[0][0]
        janela = 15
        i0 = max(0, idx_inicio - janela)
        i1 = min(len(modelo_manchas) - 1, idx_inicio + janela)
        modelo_manchas[i0:i1 + 1] = np.linspace(
            modelo_manchas[i0], modelo_manchas[i1], i1 - i0 + 1
        )

    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# ============================================================
# AJUSTES FLATTEN LOCAL POR REGIÃO (configuração completa por [t_i, t_f])
# ============================================================

print(f"\nAplicando {len(regioes_flatten)} região(ões) flatten local...")
for ini, fim, wlen, pord, sig, btol, niter in regioes_flatten:
    regiao = (t >= ini) & (t <= fim)
    if not np.any(regiao):
        print(f"  [flatten] nenhum ponto em {ini:.4f} - {fim:.4f}")
        continue

    # True = ignora no flatten. Usa somente pontos bons dentro da região.
    mask_flatten_regiao = (~regiao) | (~mask_good_flares)

    print(
        f"  [flatten] {ini:.4f} - {fim:.4f} | "
        f"window={wlen}, poly={pord}, sigma={sig}, "
        f"break_tol={btol}, niters={niter}"
    )

    flcd_r, trend_r = lc2_m.flatten(
        window_length=int(wlen),
        polyorder=int(pord),
        return_trend=True,
        break_tolerance=float(btol),
        niters=int(niter),
        sigma=float(sig),
        mask=mask_flatten_regiao
    )

    modelo_manchas[regiao] = trend_r.flux.value[regiao]
    bordas_indices.add(np.where(regiao)[0][0])
    bordas_indices.add(np.where(regiao)[0][-1])

# ============================================================
# AJUSTES SPLINE POR REGIÃO (aplicados por último)
# ============================================================

print(f"\nAplicando {len(regioes_spline)} região(ões) spline...")
for ini, fim, nbins, k, fator_s, sigma_clip_int in regioes_spline:
    idx = (t >= ini) & (t <= fim)
    if not np.any(idx):
        print(f"  [spline] nenhum ponto em {ini:.4f} - {fim:.4f}")
        continue

    print(
        f"  [spline] {ini:.4f} - {fim:.4f} | "
        f"nbins={nbins}, k={k}, fator_s={fator_s}, sigma_clip={sigma_clip_int}"
    )

    modelo_manchas[idx] = fitting_spline(
        t[idx], f[idx], mask_good_flares[idx],
        nbins=nbins, k=k, fator_s=fator_s,
        sigma_clip_interno=sigma_clip_int
    )

    bordas_indices.add(np.where(idx)[0][0])
    bordas_indices.add(np.where(idx)[0][-1])

# ============================================================
# COSTURA FINAL — junta auto + manual + flatten local + spline local
# ============================================================

# ============================================================
# LIMPEZA DE BORDAS INTERNAS (evita costura falsa dentro de regiões já cobertas)
# ============================================================

def limpar_bordas_internas(t, bordas_indices, regioes_overlay, margem=0.001):
    """
    Remove bordas que caem ESTRITAMENTE dentro de uma região de override
    (manual/flatten/spline), mantendo apenas bordas que estão fora dessas
    regiões ou exatamente nas extremidades delas.
    """
    bordas_validas = set()
    for idx in bordas_indices:
        tempo = t[idx]
        dentro_sem_ser_borda = False
        for ini, fim in regioes_overlay:
            if (tempo > ini + margem) and (tempo < fim - margem):
                dentro_sem_ser_borda = True
                break
        if not dentro_sem_ser_borda:
            bordas_validas.add(idx)
    return bordas_validas

regioes_overlay = (
    [(r[0], r[1]) for r in ajustes_manuais] +
    [(r[0], r[1]) for r in regioes_flatten] +
    [(r[0], r[1]) for r in regioes_spline]
)

n_antes = len(bordas_indices)
bordas_indices = limpar_bordas_internas(t, bordas_indices, regioes_overlay)
print(f"\nBordas internas removidas: {n_antes - len(bordas_indices)} (restaram {len(bordas_indices)})")

# ============================================================
# COSTURA FINAL — junta auto + manual + flatten local + spline local
# ============================================================

# DEPOIS — substitua pelo wrapper + chamada
def costurar_com_gaps(t, modelo, bordas, limite_gap_fator=5, tamanho_janela=32, sigma_gauss=10):
    dt_mediano = np.median(np.diff(t))
    limite_gap = dt_mediano * limite_gap_fator
    quebras = list(np.where(np.diff(t) > limite_gap)[0] + 1)
    seg_inicios = [0] + quebras
    seg_fins    = quebras + [len(t)]

    modelo_costurado = costurar_bordas(
        t, modelo, bordas,
        dados_observados=None,
        tamanho_janela=tamanho_janela,
        sigma_gauss=0  # sem gauss aqui
    )

    if sigma_gauss > 0:
        for i0, i1 in zip(seg_inicios, seg_fins):
            seg = modelo_costurado[i0:i1]
            if len(seg) > 1:
                modelo_costurado[i0:i1] = gaussian_filter1d(seg, sigma=sigma_gauss)

    return modelo_costurado

modelo_manchas_suave = costurar_bordas(
    t, modelo_manchas, sorted(bordas_indices),
    tamanho_janela=150,  # Aumentado para ~50 minutos de transição total
    sigma_gauss=20       # Leve aumento na gaussiana para estabilizar a cadência alta
)

# ============================================================
# RESÍDUO
# ============================================================

residual_manchas = f / modelo_manchas_suave
print("\nOK — Ajuste concluído.")

# ============================================================
# GRÁFICOS
# ============================================================

%matplotlib qt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

ax1.plot(
    t,
    f,
    'k.-',
    ms=1.5,
    lw=0.5,
    alpha=0.6,
    label='Dados'
)
ax1.plot(
    t,
    modelo_manchas_suave,
    color='red',
    lw=2.2,
    label='Modelo Final'
)

for ini, fim, *_ in regioes_spline:
    ax1.axvspan(ini, fim, color='dodgerblue', alpha=0.10, label='_spline')
for ini, fim, *_ in regioes_flatten:
    ax1.axvspan(ini, fim, color='mediumseagreen', alpha=0.10, label='_flatten')

ax1.set_ylabel("Fluxo")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(
    t,
    residual_manchas,
    'b.-',
    ms=1.5,
    lw=0.5,
    alpha=0.7
)
ax2.axhline(1, ls='--', alpha=0.5)

for ini, fim, *_ in regioes_spline:
    ax2.axvspan(ini, fim, color='dodgerblue', alpha=0.10)
for ini, fim, *_ in regioes_flatten:
    ax2.axvspan(ini, fim, color='mediumseagreen', alpha=0.10)

ax2.set_ylabel("Residual")
ax2.set_xlabel("Tempo [BTJD]")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()


AJUSTE: Polinomial com Segmentos + Spline em Regiões

[Cadência 20s] Janela de Suavização: 10 horas (1800 pontos).
Processando 13 segmento(s) contínuo(s) real(is)...
Ajuste automático...
Aplicando 8 ajustes manuais...

Aplicando 0 região(ões) flatten local...

Aplicando 1 região(ões) spline...
  [spline] 3899.9880 - 3900.9960 | nbins=100, k=4, fator_s=0.9, sigma_clip=1.2

Bordas internas removidas: 7 (restaram 19)

Aplicando costura ultra-suave (Blending) em 19 emendas...

OK — Ajuste concluído.


In [6]:
%matplotlib qt
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# PARAMETROS DOS PLANETAS (Plavchan et al. 2020)
# ============================================================

# AU Mic b
T0_b    = 1330.39051   # TBJD (BJD - 2457000)
P_b     = 8.463000     # dias
dur_b   = 3.50 / 24.0  # duracao em dias

# AU Mic c
T0_c    = 1342.2223    # TBJD
P_c     = 18.859019    # dias
dur_c   = 4.5  / 24.0  # duracao em dias

# ============================================================
# FUNCAO: centros de transito no intervalo de t
# ============================================================

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil( (t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

t_min, t_max = t.min(), t.max()
centers_b = transit_times(T0_b, P_b, t_min, t_max)
centers_c = transit_times(T0_c, P_c, t_min, t_max)

print(f'Transitos AU Mic b: {len(centers_b)}')
for tc in centers_b: print(f'  BTJD = {tc:.5f}')
print(f'Transitos AU Mic c: {len(centers_c)}')
for tc in centers_c: print(f'  BTJD = {tc:.5f}')

# ============================================================
# FIGURA: 2 paineis
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# ---- Painel superior: dados + modelo ----
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.5, label='Dados Originais', zorder=1)
ax1.plot(t, modelo_manchas_suave, 'r-', lw=2, label='Modelo (manchas)', zorder=3)

# Mascara de flares/transitos (CSV)
_lm = False
for ini, fim in mascara_flares_list:
    lbl = 'Flares e Transitos detectados (IV)' if not _lm else '_nolegend_'
    ax1.axvspan(ini, fim, color='orange', alpha=0.25, zorder=2, label=lbl)
    _lm = True

# Transitos AU Mic b
_lb = False
for tc in centers_b:
    lbl = 'Transito AU Mic b' if not _lb else '_nolegend_'
    ax1.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.28, zorder=2, label=lbl)
    ax1.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb = True

# Transitos AU Mic c
_lc = False
for tc in centers_c:
    lbl = 'Transito AU Mic c' if not _lc else '_nolegend_'
    ax1.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.26, zorder=2, label=lbl)
    ax1.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc = True

ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('AU Mic - Ajuste de Manchas, Mascara e Transitos Planetarios', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax1.grid(alpha=0.25)

# ---- Painel inferior: residual ----
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.6, label='Residual', zorder=1)
ax2.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.6)

_lm2 = False
for ini, fim in mascara_flares_list:
    lbl = 'Mascara' if not _lm2 else '_nolegend_'
    ax2.axvspan(ini, fim, color='orange', alpha=0.25, zorder=2, label=lbl)
    _lm2 = True

_lb2 = False
for tc in centers_b:
    lbl = 'AU Mic b' if not _lb2 else '_nolegend_'
    ax2.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.28, zorder=2, label=lbl)
    ax2.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb2 = True

_lc2 = False
for tc in centers_c:
    lbl = 'AU Mic c' if not _lc2 else '_nolegend_'
    ax2.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.26, zorder=2, label=lbl)
    ax2.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc2 = True

ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.legend(loc='upper right', fontsize=11, framealpha=0.9)
ax2.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Transitos AU Mic b: 3
  BTJD = 3886.21651
  BTJD = 3894.67951
  BTJD = 3903.14251
Transitos AU Mic c: 2
  BTJD = 3888.18986
  BTJD = 3907.04888


In [8]:
%matplotlib qt
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# PARAMETROS DOS PLANETAS (Plavchan et al. 2020 + Kanodia 2024)
# ============================================================

# AU Mic b
T0_b    = 1330.39051   # TBJD (BJD - 2457000)
P_b     = 8.463000     # dias
dur_b   = 3.50 / 24.0  # duracao em dias

# AU Mic c
T0_c    = 1342.2223    # TBJD
P_c     = 18.859019    # dias
dur_c   = 4.5  / 24.0  # duracao em dias

# AU Mic d (novo) - convertido de JD para TBJD
T0_d    = 2458333.3211 - 2457000  # = 1333.3211 TBJD
P_d     = 12.73812     # dias
dur_d   = None         # duração não disponível - só linha vertical

# ============================================================
# FUNCAO: centros de transito no intervalo de t
# ============================================================

def transit_times(T0, P, t_min, t_max):
    """Calcula todos os T0 equivalentes (T0 + n*P) que caem em [t_min, t_max]"""
    n_min = int(np.ceil( (t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

# Seu novo intervalo: 3883.02 - 3907 BTJD
t_min, t_max = t.min(), t.max()
centers_b = transit_times(T0_b, P_b, t_min, t_max)
centers_c = transit_times(T0_c, P_c, t_min, t_max)
centers_d = transit_times(T0_d, P_d, t_min, t_max)

print(f'Transitos AU Mic b: {len(centers_b)}')
for tc in centers_b: print(f'  BTJD = {tc:.5f}')
print(f'Transitos AU Mic c: {len(centers_c)}')
for tc in centers_c: print(f'  BTJD = {tc:.5f}')
print(f'Transitos AU Mic d: {len(centers_d)}')
for tc in centers_d: print(f'  BTJD = {tc:.5f}')

# ============================================================
# FIGURA: 2 paineis
# ============================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# ---- Painel superior: dados + modelo ----
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.5, label='Dados Originais', zorder=1)
ax1.plot(t, modelo_manchas_suave, 'r-', lw=2, label='Modelo (manchas)', zorder=3)

# Mascara de flares/transitos (CSV)
_lm = False
for ini, fim in mascara_flares_list:
    lbl = 'Mascara (flares/transitos)' if not _lm else '_nolegend_'
    ax1.axvspan(ini, fim, color='orange', alpha=0.25, zorder=2, label=lbl)
    _lm = True

# Transitos AU Mic b
_lb = False
for tc in centers_b:
    lbl = 'Transito AU Mic b' if not _lb else '_nolegend_'
    ax1.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.28, zorder=2, label=lbl)
    ax1.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb = True

# Transitos AU Mic c
_lc = False
for tc in centers_c:
    lbl = 'Transito AU Mic c' if not _lc else '_nolegend_'
    ax1.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.26, zorder=2, label=lbl)
    ax1.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc = True

# Transitos AU Mic d (apenas linha vertical - duração não disponível)
_ld = False
for tc in centers_d:
    lbl = 'Transito AU Mic d' if not _ld else '_nolegend_'
    ax1.axvline(tc, color='purple', lw=1.5, ls='-.', alpha=0.8, zorder=4, label=lbl)
    _ld = True

ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('AU Mic - Ajuste de Manchas, Mascara e Transitos Planetarios (b, c, d)', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax1.grid(alpha=0.25)

# ---- Painel inferior: residual ----
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.6, label='Residual', zorder=1)
ax2.axhline(1.0, color='gray', ls='--', lw=1.2, alpha=0.6)

_lm2 = False
for ini, fim in mascara_flares_list:
    lbl = 'Mascara' if not _lm2 else '_nolegend_'
    ax2.axvspan(ini, fim, color='orange', alpha=0.25, zorder=2, label=lbl)
    _lm2 = True

_lb2 = False
for tc in centers_b:
    lbl = 'AU Mic b' if not _lb2 else '_nolegend_'
    ax2.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.28, zorder=2, label=lbl)
    ax2.axvline(tc, color='royalblue', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lb2 = True

_lc2 = False
for tc in centers_c:
    lbl = 'AU Mic c' if not _lc2 else '_nolegend_'
    ax2.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.26, zorder=2, label=lbl)
    ax2.axvline(tc, color='seagreen', lw=1.2, ls='--', alpha=0.7, zorder=4)
    _lc2 = True

_ld2 = False
for tc in centers_d:
    lbl = 'AU Mic d' if not _ld2 else '_nolegend_'
    ax2.axvline(tc, color='purple', lw=1.5, ls='-.', alpha=0.8, zorder=4, label=lbl)
    _ld2 = True

ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.legend(loc='upper right', fontsize=10, framealpha=0.9)
ax2.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Transitos AU Mic b: 3
  BTJD = 3886.21651
  BTJD = 3894.67951
  BTJD = 3903.14251
Transitos AU Mic c: 2
  BTJD = 3888.18986
  BTJD = 3907.04888
Transitos AU Mic d: 2
  BTJD = 3893.68322
  BTJD = 3906.42134


In [66]:


%matplotlib qt
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Gráfico 1: Dados originais + manchas
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas, 'r-', linewidth=2.5, label='Ajuste (Manchas)')
ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('Passo 1: Ajuste de Manchas Estelares', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(alpha=0.3)





In [67]:
# ============================================================
# PASSO 3: VISUALIZAR RESIDUAIS E SELECIONAR FLARES INTERATIVAMENTE
# ============================================================
print("\n" + "="*60)
print("VISUALIZAÇÃO: Residuais após manchas (para seleção de flares)")
print("="*60)

%matplotlib qt
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Gráfico 1: Dados originais + manchas
ax1.plot(t, f, 'k.-', ms=1.5, lw=0.5, alpha=0.6, label='Dados Originais')
ax1.plot(t, modelo_manchas, 'r-', linewidth=2.5, label='Ajuste (Manchas)')
ax1.set_ylabel('Fluxo Normalizado', fontsize=14)
ax1.set_title('Passo 1: Ajuste de Manchas Estelares', fontsize=15, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(alpha=0.3)

# Gráfico 2: Residuais para detecção de flares
ax2.plot(t, residual_manchas, 'b.-', ms=1.5, lw=0.5, alpha=0.7, label='Residual (original/manchas)')
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Fluxo Residual', fontsize=14)
ax2.set_xlabel('Tempo [BTJD dias]', fontsize=14)
ax2.set_title('Residuais para Seleção de Flares e Trânsitos', fontsize=15, fontweight='bold')
ax2.legend(loc='upper right', fontsize=12)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n→ Feche o gráfico e continue na próxima célula para inserir os intervalos de flares/trânsitos")



VISUALIZAÇÃO: Residuais após manchas (para seleção de flares)

→ Feche o gráfico e continue na próxima célula para inserir os intervalos de flares/trânsitos


In [ ]:
# ============================================================
# MEDIA E DESVIO PADRAO COM MASCARA
# ============================================================
from astropy.stats import sigma_clipped_stats

# ---- Parâmetros AU Mic d (definidos aqui pois célula 4 não tem) ----
T0_d   = 2458333.3211 - 2457000   # = 1333.3211 TBJD
P_d    = 12.73596                  # dias
dur_d  = None                      # duração não disponível

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil( (t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

centers_d = transit_times(T0_d, P_d, t.min(), t.max())
print(f"Trânsitos AU Mic d: {len(centers_d)}")
for tc in centers_d: print(f"  BTJD = {tc:.5f}")

# ---- Passo 1: filtrar pontos bons ----
x_detrend = []
y_detrend = []
for j in range(len(mask_good_flares)):
    if mask_good_flares[j] == True:
        y_detrend.append(residual_manchas[j])
        x_detrend.append(t[j])

mascara2 = [False] * len(mask_good_flares)

x_detrend = np.array(x_detrend)
y_detrend = np.array(y_detrend)

print(f"\nPontos totais:     {len(t)}")
print(f"Pontos bons:       {len(y_detrend)}")
print(f"Pontos mascarados: {len(t) - len(y_detrend)}")

# ---- Passo 2: média e desvio padrão iniciais ----
y_mean  = np.mean(y_detrend)
y_mean2 = [y_mean] * len(y_detrend)
desvio  = np.std(y_detrend)

med_desv2 = y_mean + 2 * desvio
med_desv3 = y_mean - 2 * desvio

print(f"\n--- 1ª iteração ---")
print(f"Média:                {y_mean:.6f}")
print(f"Desvio padrão:        {desvio:.6f}")
print(f"Corte superior (+2σ): {med_desv2:.6f}")
print(f"Corte inferior (-2σ): {med_desv3:.6f}")

# ---- Passo 3: corte em ±2σ ----
xdefinitivo = []
ydefinitivo = []
for j in range(len(y_detrend)):
    if y_detrend[j] < med_desv2 and y_detrend[j] > med_desv3:
        ydefinitivo.append(y_detrend[j])
        xdefinitivo.append(x_detrend[j])

xdefinitivo = np.array(xdefinitivo)
ydefinitivo = np.array(ydefinitivo)

print(f"\nPontos após corte ±2σ: {len(ydefinitivo)}")

# ---- Passo 4: sigma-clipping iterativo ----
y_m, _, desvio2 = sigma_clipped_stats(ydefinitivo, sigma=3, maxiters=5)
med2     = y_m + 3 * desvio2
med2_inf = y_m - 3 * desvio2

print(f"\n--- 2ª iteração (sigma-clipping 3σ iterativo) ---")
print(f"Média clipped:  {y_m:.6f}")
print(f"Desvio padrão:  {desvio2:.6f}")
print(f"Limiar +3σ:     {med2:.6f}")
print(f"Limiar -3σ:     {med2_inf:.6f}")

# ---- Passo 5: gráfico ----
%matplotlib qt

fig, ax = plt.subplots(1, 1, figsize=(16, 5))

# Residual completo
l1, = ax.plot(t, residual_manchas, 'k.-', ms=1.5, lw=0.5, alpha=0.5, zorder=1)

# Pontos usados no cálculo
l2, = ax.plot(xdefinitivo, ydefinitivo, 'b.', ms=1.5, alpha=0.6, zorder=2)

# Linhas de referência
l3 = ax.axhline(y_m,       color='green',  lw=1.5, ls='-',  alpha=0.9, zorder=5)
l4 = ax.axhline(med2,      color='red',    lw=1.5, ls='--', alpha=0.9, zorder=5)
l5 = ax.axhline(med2_inf,  color='red',    lw=1.5, ls=':',  alpha=0.9, zorder=5)
l6 = ax.axhline(
    med_desv2,
    color='magenta',
    lw=1.5,
    ls='--'
)

l7 = ax.axhline(
    med_desv3,
    color='magenta',
    lw=1.5,
    ls=':'
)

# Máscara do CSV
p_mask = None
for ini, fim in mascara_flares_list:
    p_mask = ax.axvspan(ini, fim, color='orange', alpha=0.20, zorder=0)

# Trânsitos AU Mic b
p_b = None
for tc in centers_b:
    p_b = ax.axvspan(tc - dur_b/2, tc + dur_b/2, color='royalblue', alpha=0.18, zorder=0)
    ax.axvline(tc, color='royalblue', lw=1.0, ls='--', alpha=0.6)

# Trânsitos AU Mic c
p_c = None
for tc in centers_c:
    p_c = ax.axvspan(tc - dur_c/2, tc + dur_c/2, color='seagreen', alpha=0.18, zorder=0)
    ax.axvline(tc, color='seagreen', lw=1.0, ls='--', alpha=0.6)

# Trânsitos AU Mic d (só linha vertical - sem duração)
l_d = None
for tc in centers_d:
    l_d = ax.axvline(tc, color='purple', lw=1.2, ls='-.', alpha=0.7)

# Legenda manual
legend_handles = [l1, l2, l3, l4, l5, l6, l7]
legend_labels  = [
    'Residual completo',
    'Pontos usados (fora máscara, ±2σ)',
    f'Média = {y_m:.5f}',
    f'+3σ  = {med2:.5f}',
    f'-3σ  = {med2_inf:.5f}',
    f'+2σ corte = {med_desv2:.5f}',
    f'-2σ corte = {med_desv3:.5f}',
]

if p_mask is not None:
    legend_handles.append(p_mask)
    legend_labels.append('Máscara CSV')
if p_b is not None:
    legend_handles.append(p_b)
    legend_labels.append('AU Mic b')
if p_c is not None:
    legend_handles.append(p_c)
    legend_labels.append('AU Mic c')
if l_d is not None:
    legend_handles.append(l_d)
    legend_labels.append('AU Mic d')

ax.legend(legend_handles, legend_labels,
          loc='upper right', fontsize=9, framealpha=0.95, ncol=2)

ax.set_xlabel('Tempo - 2457000 [BTJD dias]', fontsize=13)
ax.set_ylabel('Fluxo Residual Normalizado', fontsize=13)
ax.set_title('AU Mic – Residual com Máscara, Limiares e Trânsitos', fontsize=14, fontweight='bold')
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Trânsitos AU Mic d: 2
  BTJD = 3893.24906
  BTJD = 3905.98502

Pontos totais:     91037
Pontos bons:       79322
Pontos mascarados: 11715

--- 1ª iteração ---
Média:                1.000013
Desvio padrão:        0.000833
Corte superior (+2σ): 1.001679
Corte inferior (-2σ): 0.998348

Pontos após corte ±2σ: 75739

--- 2ª iteração (sigma-clipping 3σ iterativo) ---
Média clipped:  0.999985
Desvio padrão:  0.000659
Limiar +3σ:     1.001963
Limiar -3σ:     0.998007


In [22]:
from astropy.stats import sigma_clipped_stats
from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt

# ---- Parâmetros AU Mic d ----
T0_d   = 2458333.3211 - 2457000
P_d    = 12.73596
dur_d  = None

def transit_times(T0, P, t_min, t_max):
    n_min = int(np.ceil((t_min - T0) / P))
    n_max = int(np.floor((t_max - T0) / P))
    return [T0 + n * P for n in range(n_min, n_max + 1)]

centers_d = transit_times(T0_d, P_d, t.min(), t.max())

print(f"Trânsitos AU Mic d: {len(centers_d)}")
for tc in centers_d:
    print(f"  BTJD = {tc:.5f}")

# ============================================================
# FILTRAGEM DOS PONTOS BONS
# ============================================================

x_detrend = []
y_detrend = []

for j in range(len(mask_good_flares)):
    if mask_good_flares[j]:
        y_detrend.append(residual_manchas[j])
        x_detrend.append(t[j])

x_detrend = np.array(x_detrend)
y_detrend = np.array(y_detrend)

print(f"\nPontos totais:     {len(t)}")
print(f"Pontos bons:       {len(y_detrend)}")
print(f"Pontos mascarados: {len(t)-len(y_detrend)}")

# ============================================================
# MÉDIA E DESVIO INICIAIS
# ============================================================

y_mean = np.mean(y_detrend)
desvio = np.std(y_detrend)

med_desv2 = y_mean + 2 * desvio
med_desv3 = y_mean - 2 * desvio

print(f"\n--- 1ª iteração ---")
print(f"Média:                {y_mean:.6f}")
print(f"Desvio padrão:        {desvio:.6f}")
print(f"Corte superior (+2σ): {med_desv2:.6f}")
print(f"Corte inferior (-2σ): {med_desv3:.6f}")

# ============================================================
# CORTE ±2σ
# ============================================================

xdefinitivo = []
ydefinitivo = []

for j in range(len(y_detrend)):
    if med_desv3 < y_detrend[j] < med_desv2:
        ydefinitivo.append(y_detrend[j])
        xdefinitivo.append(x_detrend[j])

xdefinitivo = np.array(xdefinitivo)
ydefinitivo = np.array(ydefinitivo)

print(f"\nPontos após corte ±2σ: {len(ydefinitivo)}")

# ============================================================
# SIGMA CLIPPING
# ============================================================

y_m, _, desvio2 = sigma_clipped_stats(
    ydefinitivo,
    sigma=3,
    maxiters=5
)

med2 = y_m + 3 * desvio2
med2_inf = y_m - 3 * desvio2

print(f"\n--- 2ª iteração (sigma-clipping 3σ iterativo) ---")
print(f"Média clipped:  {y_m:.6f}")
print(f"Desvio padrão:  {desvio2:.6f}")
print(f"Limiar +3σ:     {med2:.6f}")
print(f"Limiar -3σ:     {med2_inf:.6f}")

# ============================================================
# DETECÇÃO DOS PICOS
# ============================================================

peaks, properties = find_peaks(y_detrend, height=med2)

print(f"\nNúmero de picos encontrados: {len(peaks)}")

for i in peaks:
    print(f"Tempo = {x_detrend[i]:.6f}   Fluxo = {y_detrend[i]:.6f}")

# ============================================================
# GRÁFICO
# ============================================================

%matplotlib qt

fig, ax = plt.subplots(1, 1, figsize=(16, 5))

# Residual completo
l1, = ax.plot(
    t,
    residual_manchas,
    'k.-',
    ms=1.5,
    lw=0.5,
    alpha=0.5,
    zorder=1
)

# Pontos usados no cálculo
l2, = ax.plot(
    xdefinitivo,
    ydefinitivo,
    'b.',
    ms=1.5,
    alpha=0.6,
    zorder=2
)

# Picos detectados
l_peaks, = ax.plot(
    x_detrend[peaks],
    y_detrend[peaks],
    marker='x',
    linestyle='None',
    color='deeppink',
    ms=8,
    mew=2,
    zorder=20
)

# Linhas de referência
l3 = ax.axhline(
    y_m,
    color='green',
    lw=1.5,
    ls='-',
    alpha=0.9,
    zorder=5
)

l4 = ax.axhline(
    med2,
    color='darkorange',
    lw=1.5,
    ls='--',
    alpha=0.9,
    zorder=5
)

l5 = ax.axhline(
    med2_inf,
    color='darkorange',
    lw=1.5,
    ls=':',
    alpha=0.9,
    zorder=5
)

l6 = ax.axhline(
    med_desv2,
    color='darkorange',
    lw=1.2,
    ls='--',
    alpha=0.8,
    zorder=5
)

l7 = ax.axhline(
    med_desv3,
    color='darkviolet',
    lw=1.2,
    ls=':',
    alpha=0.8,
    zorder=5
)

# Máscara CSV
p_mask = None
for ini, fim in mascara_flares_list:
    p_mask = ax.axvspan(
        ini,
        fim,
        color='gold',
        alpha=0.20,
        zorder=0
    )

# AU Mic b
p_b = None
for tc in centers_b:
    p_b = ax.axvspan(
        tc-dur_b/2,
        tc+dur_b/2,
        color='royalblue',
        alpha=0.18,
        zorder=0
    )
    ax.axvline(tc, color='royalblue', lw=1.0, ls='--', alpha=0.6)

# AU Mic c
p_c = None
for tc in centers_c:
    p_c = ax.axvspan(
        tc-dur_c/2,
        tc+dur_c/2,
        color='seagreen',
        alpha=0.18,
        zorder=0
    )
    ax.axvline(tc, color='seagreen', lw=1.0, ls='--', alpha=0.6)

# AU Mic d
l_d = None
for tc in centers_d:
    l_d = ax.axvline(
        tc,
        color='purple',
        lw=1.2,
        ls='-.',
        alpha=0.7
    )

# Legenda
legend_handles = [l1, l2, l_peaks, l3, l4, l5, l6, l7]

legend_labels = [
    'Residual completo',
    'Pontos usados (fora máscara, ±2σ)',
    f'Picos > +3σ ({len(peaks)})',
    f'Média = {y_m:.5f}',
    f'+3σ = {med2:.5f}',
    f'-3σ = {med2_inf:.5f}',
    f'+2σ corte = {med_desv2:.5f}',
    f'-2σ corte = {med_desv3:.5f}',
]

if p_mask is not None:
    legend_handles.append(p_mask)
    legend_labels.append('Máscara CSV')

if p_b is not None:
    legend_handles.append(p_b)
    legend_labels.append('AU Mic b')

if p_c is not None:
    legend_handles.append(p_c)
    legend_labels.append('AU Mic c')

if l_d is not None:
    legend_handles.append(l_d)
    legend_labels.append('AU Mic d')

ax.legend(
    legend_handles,
    legend_labels,
    loc='upper right',
    fontsize=9,
    framealpha=0.95,
    ncol=2
)

ax.set_xlabel('Tempo - 2457000 [BTJD dias]', fontsize=13)
ax.set_ylabel('Fluxo Residual Normalizado', fontsize=13)
ax.set_title(
    'AU Mic – Residual com Máscara, Limiares, Trânsitos e Picos',
    fontsize=14,
    fontweight='bold'
)

ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

Trânsitos AU Mic d: 2
  BTJD = 3893.24906
  BTJD = 3905.98502

Pontos totais:     91037
Pontos bons:       79322
Pontos mascarados: 11715

--- 1ª iteração ---
Média:                1.000013
Desvio padrão:        0.000833
Corte superior (+2σ): 1.001679
Corte inferior (-2σ): 0.998348

Pontos após corte ±2σ: 75739

--- 2ª iteração (sigma-clipping 3σ iterativo) ---
Média clipped:  0.999985
Desvio padrão:  0.000659
Limiar +3σ:     1.001963
Limiar -3σ:     0.998007

Número de picos encontrados: 1078
Tempo = 3883.120561   Fluxo = 1.002331
Tempo = 3883.453896   Fluxo = 1.002731
Tempo = 3883.464545   Fluxo = 1.002020
Tempo = 3883.539545   Fluxo = 1.002002
Tempo = 3883.686537   Fluxo = 1.002858
Tempo = 3883.849500   Fluxo = 1.001967
Tempo = 3883.944408   Fluxo = 1.004420
Tempo = 3883.945566   Fluxo = 1.002983
Tempo = 3883.946029   Fluxo = 1.002095
Tempo = 3883.995566   Fluxo = 1.002053
Tempo = 3884.031214   Fluxo = 1.002146
Tempo = 3884.188391   Fluxo = 1.002157
Tempo = 3884.232373   Fluxo = 1.0